# Build the per-patient radiomic vectors

Run in the **radiomics** or **modeling** environment (pandas only).
Inputs are the raw CSVs written by `pyradiomics_features_EGD_normal_tissue_contrasting.ipynb`.


In [ ]:
import pandas as pd

In [ ]:

df = pd.read_csv("../../radiomics_features.csv")

## Tumour + mirrored ROI vector

Produces `radiomics_embeddings/t1t2flair_mirror_radiomics_features_EGD.csv`
(one row per patient, `{T1,T2,FLAIR}_{original,mirrored}_{filter}_{family}_{feature}`).


In [ ]:
# One row per patient: the extraction wrote one row per contrast (T1, T2, FLAIR),
# each filling only its own block of columns. Keep the first non-null value per column.
df_mirror = df.groupby("Patient", sort=False).first().reset_index()
df_mirror.to_csv("../radiomics_embeddings/t1t2flair_mirror_radiomics_features_EGD.csv", index=False)
df_mirror.shape

In [ ]:
df.shape

In [ ]:
total_columns = df.shape[1]

# Calculate the midpoint
splitpoint = total_columns // 3 
splitpoint

In [ ]:
df1 = df.iloc[:, :splitpoint+1]
df2 = df.iloc[:, splitpoint+1:splitpoint*2+1]
df3 = df.iloc[:, splitpoint*2+1:]

In [ ]:
df1

In [ ]:
df2

In [ ]:
df3

In [ ]:
df

## Boundary-shell vector

Produces `radiomics_embeddings/border_features_EGD.csv` (one row per patient, columns renamed to
`{T1,T2,FLAIR}_border_...`). Row order matches the mirrored CSV, which `get_data.py` relies on.


In [ ]:
import pandas as pd
df = pd.read_csv("../../border_radiomics_features_EGD.csv")

total_columns = df.shape[1]

# Calculate the midpoint
splitpoint = total_columns // 3 

df_patient = df.loc[~df["Patient"].duplicated(keep="first")].reset_index(drop=True)
df = df.drop(columns=['Patient'])

df1 = df.iloc[:, :splitpoint]
df2 = df.iloc[:, splitpoint:splitpoint*2]
df3 = df.iloc[:, splitpoint*2:]

# Remove rows where all elements are NaN
df1 = df1.dropna(how="all")
df1 = df1.reset_index(drop=True)
df2 = df2.dropna(how="all")
df2 = df2.reset_index(drop=True)
df3 = df3.dropna(how="all")
df3 = df3.reset_index(drop=True)

In [ ]:
df_concat = pd.concat([df1, df2, df3], axis=1)
df_concat.columns = df_concat.columns.str.replace("_original_", "_border_", n=1)
#df_concat.insert(0, "Patient", df_patient["Patient"])

In [ ]:
df_concat.columns

In [ ]:
df_concat.to_csv("../radiomics_embeddings/border_features_EGD.csv", index=False)